# Ecological Overshoot Analysis

Top 10 and bottom 10 countries across key ecological footprint and biocapacity metrics, with visualizations.

**Input Data:** `data/overshoot_results.csv` and `data/overshoot_global_summary.csv`, generated by `scripts/calculate_overshoot.py` from FAOSTAT bulk downloads.

---

### Glossary of Abbreviations

| Abbreviation | Full Term | Definition |
|---|---|---|
| **EF** | Ecological Footprint | Total biologically productive area required to produce the resources a population consumes and absorb the CO₂ waste it generates, measured in gha |
| **BC** | Biocapacity | Total biologically productive area available within a country to regenerate resources and absorb waste, measured in gha |
| **gha** | Global Hectare | A productivity-weighted hectare that represents world-average biological productivity; the common unit for comparing EF and BC across land types |
| **EQF** | Equivalence Factor | Converts physical hectares of a specific land type (e.g., cropland, forest) into gha, reflecting that land type's relative productivity versus the world average |
| **YF** | Yield Factor | Ratio of national yield to world-average yield for a given land type; captures how productive a country's land is compared to the global mean |
| **PPR** | Primary Production Required | Total marine phytoplankton production needed to sustain a given fish catch, derived via the trophic-level method |
| **TL** | Trophic Level | Position in the marine food chain (1 = phytoplankton, 2 = herbivores, 3+ = predators); higher TL requires exponentially more primary production per tonne of catch |
| **TE** | Transfer Efficiency | Fraction of energy transferred between trophic levels; 0.1013 (approx. 10%) per level (Pauly & Christensen 1995) |
| **AFCS** | Annual Forest Carbon Sequestration | Average rate at which forests absorb carbon: 0.73 t C/ha/yr, equivalent to 2.68 t CO₂/ha/yr (IPCC 2006) |
| **NAI** | Net Annual Increment | World-average annual growth rate of forest biomass: 1.81 m³/ha/yr (FAO Global Forest Resources Assessment) |
| **NPP** | Net Primary Productivity | Annual plant biomass production per unit area; for grassland we use 2.45 t DM/ha/yr (Haberl et al. 2007) |
| **DM** | Dry Matter | Biomass weight excluding water; the standard unit for livestock feed demand and grassland productivity |
| **FBS** | Food Balance Sheet | FAOSTAT dataset tracking national food production, imports, exports, and utilisation by commodity |
| **FAOSTAT** | FAO Statistical Database | The United Nations Food and Agriculture Organization's open database of agricultural, land-use, trade, and emissions statistics |
| **GLEAM** | Global Livestock Environmental Assessment Model | FAO model providing feed requirements (including pasture DM demand) per livestock species and production system |
| **GFN** | Global Footprint Network | Organisation publishing the official National Footprint and Biocapacity Accounts |
| **CO₂** | Carbon Dioxide | Greenhouse gas; the carbon EF component converts territorial CO₂ emissions into the forest area needed for sequestration |
| **bn** | Billion | 10⁹ |
| **cap** | Per Capita | Per person |
| **kt** | Kilotonnes | 1,000 metric tonnes |
| **ha** | Hectare | 10,000 m² (approximately 2.47 acres) |

### EF Component Formulas (Quick Reference)

| Component | EF Formula | EQF | Key Data Source |
|---|---|---|---|
| **Cropland** | `EF = Sum(Production_t / WorldYield_t/ha) x EQF` | 2.1 | `crop_production.csv` (FAOSTAT Production / Area harvested) |
| **Grazing** | `EF = Sum(Heads x PastureDM_t/head) / NPP x EQF` | 0.5 | `livestock_stocks.csv` + GLEAM coefficients |
| **Forest** | `EF = Roundwood_m3 / NAI x EQF` | 1.3 | `timber_production.csv` (FAOSTAT Forestry Production) |
| **Fishing** | `EF = PPR / MarineYield x EQF` | 0.4 | `fish_supply_fbs.csv` (FAOSTAT Food Balance Sheets) |
| **Built-up** | `EF = BC = BuiltUpArea_ha x EQF` | 2.2 | `land_use` data (FAOSTAT Land Use domain) |
| **Carbon** | `EF = CO2_t x 0.65 / (AFCS x 44/12) x EQF` | 1.3 | `co2_energy_faostat.csv` (FAOSTAT Emissions) |

**Biocapacity** for each land type: `BC = NationalArea_ha x YF x EQF`

**Ecological Deficit** = BC_total - EF_total (negative value = deficit, positive = reserve)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

ROOT = Path("__file__").resolve().parent if "__file__" in dir() else Path(".").resolve().parent
DATA = ROOT / "data"

df = pd.read_csv(DATA / "overshoot_results.csv")
gs = pd.read_csv(DATA / "overshoot_global_summary.csv")

# Add derived columns
df["deficit_per_capita"] = df["ecological_deficit_gha"] / (df["population"] * 1000)

# Latest year slice (for rankings)
YEAR = df["year"].max()
latest = df[df["year"] == YEAR].copy()

# Filter out very small territories (pop < 10k = 10 in thousands) for per-capita rankings
latest_big = latest[latest["population"] > 10].copy()

print(f"Data: {len(df)} rows, {df['area_code'].nunique()} countries, years {df['year'].min()}-{df['year'].max()}")
print(f"Rankings based on: {YEAR}  (countries with pop > 10,000)")

---
## 1. Global Overshoot Trend

**What this shows:** Three panels tracking world-level overshoot from 2014 to 2023.

**Formulas:**
- **Panel 1 (EF vs BC):** Total world EF (sum of all 6 components across all countries) and total world BC (sum of 5 components; carbon has no BC), both in billion gha.
- **Panel 2 (Number of Earths):** `Number of Earths = World EF / World BC`. A value of 1.0 means humanity uses exactly one Earth's regenerative capacity. Values above 1.0 indicate overshoot.
- **Panel 3 (Overshoot Day):** `Overshoot Day = floor(365 x World BC / World EF)`. The calendar date by which humanity has used nature's entire annual budget. Earlier dates indicate greater overshoot.

**Input data:** `data/overshoot_global_summary.csv` — columns: `year`, `world_ef_gha`, `world_bc_gha`, `number_of_earths`, `overshoot_day` (day-of-year integer).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1a: World EF vs BC
ax = axes[0]
ax.plot(gs["year"], gs["world_ef_gha"] / 1e9, "o-", color="#d62728", linewidth=2, label="Ecological Footprint")
ax.plot(gs["year"], gs["world_bc_gha"] / 1e9, "s-", color="#2ca02c", linewidth=2, label="Biocapacity")
ax.fill_between(gs["year"], gs["world_bc_gha"] / 1e9, gs["world_ef_gha"] / 1e9, alpha=0.15, color="#d62728")
ax.set_ylabel("Billion global hectares")
ax.set_title("World EF vs Biocapacity")
ax.legend()

# 1b: Number of Earths
ax = axes[1]
ax.bar(gs["year"], gs["number_of_earths"], color="#ff7f0e", edgecolor="white")
ax.axhline(1.0, color="#2ca02c", linestyle="--", linewidth=1.5, label="1 Earth")
ax.set_ylabel("Number of Earths")
ax.set_title("Number of Earths Required")
ax.legend()

# 1c: Overshoot Day
ax = axes[2]
from datetime import datetime, timedelta
dates = [datetime(int(y), 1, 1) + timedelta(days=int(d) - 1) for y, d in zip(gs["year"], gs["overshoot_day"])]
day_of_year = gs["overshoot_day"].values
ax.plot(gs["year"], day_of_year, "D-", color="#9467bd", linewidth=2, markersize=8)
ax.set_ylabel("Day of year")
ax.set_title("Earth Overshoot Day")
ax.invert_yaxis()
# Add month labels on right
for y, d in zip(gs["year"], dates):
    ax.annotate(d.strftime("%b %d"), (y, int(d.strftime("%j"))), fontsize=8, ha="center", va="bottom")

for ax in axes:
    ax.set_xlabel("Year")

fig.suptitle("Global Ecological Overshoot (2014-2023)", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

---
## 2. World Footprint Composition (2023)

**What this shows:** Two pie charts decomposing the global EF and BC into their constituent land-type components for the latest year.

**EF composition (6 components):**
- **Cropland** — area needed to grow crops consumed: `Sum(Production / WorldYield) x EQF_cropland(2.1)`
- **Grazing** — pasture area needed for livestock feed: `Sum(Heads x PastureDM) / NPP x EQF_grazing(0.5)`
- **Forest** — area to sustain roundwood harvest: `Roundwood / NAI x EQF_forest(1.3)`
- **Fishing** — marine area to sustain fish catch via PPR method: `PPR / MarineYield x EQF_fishing(0.4)`
- **Built-up** — land covered by infrastructure: `BuiltUpArea x EQF_builtup(2.2)`
- **Carbon** — forest area to sequester CO₂ emissions: `CO₂ x 0.65 / 2.68 x EQF_forest(1.3)`

**BC composition (5 components):** Same land types except carbon (forest BC already represents sequestration capacity). Each: `NationalArea x YF x EQF`.

**Input data:** `data/overshoot_results.csv` — columns `ef_{component}_gha` and `bc_{component}_gha`, summed across all countries for the latest year.

In [ ]:
components = ["cropland", "grazing", "forest", "fishing", "built_up", "carbon"]
labels = ["Cropland", "Grazing", "Forest", "Fishing", "Built-up", "Carbon"]
colors_ef = ["#2ca02c", "#8c564b", "#1f77b4", "#17becf", "#7f7f7f", "#d62728"]
colors_bc = ["#2ca02c", "#8c564b", "#1f77b4", "#17becf", "#7f7f7f"]

w = latest.drop(columns=["area_code", "area", "year"], errors="ignore")
ef_vals = [w[f"ef_{c}_gha"].sum() / 1e9 for c in components]
bc_components = [c for c in components if c != "carbon"]
bc_vals = [w[f"bc_{c}_gha"].sum() / 1e9 for c in bc_components]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# EF pie
ax = axes[0]
wedges, texts, autotexts = ax.pie(ef_vals, labels=labels, colors=colors_ef, autopct="%1.0f%%",
                                   startangle=90, pctdistance=0.8)
for t in autotexts:
    t.set_fontsize(9)
ax.set_title(f"Ecological Footprint\n{sum(ef_vals):.1f} bn gha", fontweight="bold")

# BC pie
ax = axes[1]
bc_labels = [l for l in labels if l != "Carbon"]
wedges, texts, autotexts = ax.pie(bc_vals, labels=bc_labels, colors=colors_bc, autopct="%1.0f%%",
                                   startangle=90, pctdistance=0.8)
for t in autotexts:
    t.set_fontsize(9)
ax.set_title(f"Biocapacity\n{sum(bc_vals):.1f} bn gha", fontweight="bold")

fig.suptitle(f"World Footprint Composition ({YEAR})", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

---
## 3. Top 10 & Bottom 10: Ecological Footprint per Capita

**What this shows:** Countries with the highest and lowest per-person resource demand, measured in gha per person.

**Formula:** `EF per capita = EF_total_gha / (population x 1000)` where population is in thousands (FAOSTAT unit).

`EF_total_gha` is the sum of all 6 EF components (cropland + grazing + forest + fishing + built-up + carbon) for that country and year.

**Input data:** `data/overshoot_results.csv` — columns: `ef_per_capita_gha`, `population` (in thousands). Countries with population below 10,000 are excluded to avoid distortion by micro-states.

**Interpretation:** Higher EF/cap means the country's residents demand more biologically productive area per person. The global average is approximately 2.7 gha/person.

In [ ]:
def plot_top_bottom(data, col, title, unit="gha/person", top_color="#d62728", bot_color="#2ca02c", n=10):
    """Horizontal bar chart showing top N and bottom N countries."""
    top = data.nlargest(n, col)[["area", col]].reset_index(drop=True)
    bot = data[data[col] > 0].nsmallest(n, col)[["area", col]].reset_index(drop=True)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    ax = axes[0]
    ax.barh(top["area"][::-1], top[col][::-1], color=top_color, edgecolor="white")
    ax.set_xlabel(unit)
    ax.set_title(f"Top {n} (highest)")
    for i, v in enumerate(top[col][::-1]):
        ax.text(v + max(top[col]) * 0.01, i, f"{v:.2f}", va="center", fontsize=9)

    ax = axes[1]
    ax.barh(bot["area"][::-1], bot[col][::-1], color=bot_color, edgecolor="white")
    ax.set_xlabel(unit)
    ax.set_title(f"Bottom {n} (lowest)")
    for i, v in enumerate(bot[col][::-1]):
        ax.text(v + max(bot[col]) * 0.02, i, f"{v:.2f}", va="center", fontsize=9)

    fig.suptitle(f"{title} ({YEAR})", fontsize=13, fontweight="bold", y=1.02)
    fig.tight_layout()
    plt.show()

    return top, bot


top_ef, bot_ef = plot_top_bottom(latest_big, "ef_per_capita_gha",
                                  "Ecological Footprint per Capita")

In [ ]:
print("Top 10 EF per capita:")
display(top_ef.style.format({"ef_per_capita_gha": "{:.2f}"}))
print("\nBottom 10 EF per capita:")
display(bot_ef.style.format({"ef_per_capita_gha": "{:.2f}"}))

---
## 4. Top 10 & Bottom 10: Biocapacity per Capita

**What this shows:** Countries with the highest and lowest per-person ecological supply (regenerative capacity of their territory).

**Formula:** `BC per capita = BC_total_gha / (population x 1000)`

`BC_total_gha = BC_cropland + BC_grazing + BC_forest + BC_fishing + BC_builtup`, where each component = `NationalArea_ha x YF x EQF`.

- **YF (Yield Factor)** = national yield / world yield for that land type; captures how productive the country's land is relative to global average
- **EQF (Equivalence Factor)** converts land-type-specific hectares to globally comparable gha

**Input data:** `data/overshoot_results.csv` — column: `bc_per_capita_gha`. Underlying area data from FAOSTAT Land Use domain (`cropland_area.csv`, `pasture_area.csv`, `forest_area.csv`).

**Interpretation:** Higher BC/cap means the country has more biologically productive land per person. Countries with large territories and small populations (e.g., Guyana, Canada, Australia) tend to rank highest.

In [ ]:
top_bc, bot_bc = plot_top_bottom(latest_big, "bc_per_capita_gha",
                                  "Biocapacity per Capita",
                                  top_color="#2ca02c", bot_color="#d62728")

In [ ]:
print("Top 10 BC per capita (most biocapacity):")
display(top_bc.style.format({"bc_per_capita_gha": "{:.2f}"}))
print("\nBottom 10 BC per capita (least biocapacity):")
display(bot_bc.style.format({"bc_per_capita_gha": "{:.2f}"}))

---
## 5. Top 10 & Bottom 10: Ecological Deficit per Capita

**What this shows:** Countries with the largest ecological deficits (consuming far more than their territory can regenerate) and the largest ecological reserves (consuming well within their means).

**Formula:** `Ecological Deficit per capita = (BC_total - EF_total) / (population x 1000)`

- **Negative value** = ecological deficit: the country's demand exceeds its own biocapacity. It relies on imports, liquidation of domestic stocks, or use of the global commons (atmosphere, oceans).
- **Positive value** = ecological reserve: the country's biocapacity exceeds its footprint.

**Input data:** `data/overshoot_results.csv` — columns: `ecological_deficit_gha` (= BC_total - EF_total, in gha), `population` (in thousands). The derived column `deficit_per_capita` is computed in this notebook.

In [ ]:
# Largest deficits (most negative)
worst_deficit = latest_big.nsmallest(10, "deficit_per_capita")[["area", "deficit_per_capita"]].reset_index(drop=True)
# Largest reserves (most positive)
best_reserve = latest_big.nlargest(10, "deficit_per_capita")[["area", "deficit_per_capita"]].reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
bars = ax.barh(worst_deficit["area"][::-1], worst_deficit["deficit_per_capita"][::-1],
               color="#d62728", edgecolor="white")
ax.set_xlabel("gha/person")
ax.set_title("10 Largest Ecological Deficits")
ax.axvline(0, color="black", linewidth=0.8)
for i, v in enumerate(worst_deficit["deficit_per_capita"][::-1]):
    ax.text(v - 0.3, i, f"{v:.1f}", va="center", ha="right", fontsize=9, color="white", fontweight="bold")

ax = axes[1]
ax.barh(best_reserve["area"][::-1], best_reserve["deficit_per_capita"][::-1],
        color="#2ca02c", edgecolor="white")
ax.set_xlabel("gha/person")
ax.set_title("10 Largest Ecological Reserves")
ax.axvline(0, color="black", linewidth=0.8)
for i, v in enumerate(best_reserve["deficit_per_capita"][::-1]):
    ax.text(v + max(best_reserve["deficit_per_capita"]) * 0.01, i, f"+{v:.1f}", va="center", fontsize=9)

fig.suptitle(f"Ecological Deficit per Capita ({YEAR})", fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
print("10 largest deficits per capita:")
display(worst_deficit.style.format({"deficit_per_capita": "{:.2f}"}))
print("\n10 largest reserves per capita:")
display(best_reserve.style.format({"deficit_per_capita": "{:+.2f}"}))

---
## 6. Top 10 & Bottom 10: Total Ecological Footprint (absolute)

**What this shows:** Countries with the largest and smallest total EF in absolute terms (not per capita). This reflects the combined effect of population size and per-person consumption.

**Formula:** `EF_total_gha = ef_cropland + ef_grazing + ef_forest + ef_fishing + ef_built_up + ef_carbon` (all in gha).

Large, populous countries with high consumption levels (China, USA, India) dominate the absolute ranking even if their per-capita values differ.

**Input data:** `data/overshoot_results.csv` — column: `ef_total_gha`. Left panel in billion gha (10⁹), right panel in million gha (10⁶) due to scale difference.

In [ ]:
top_abs = latest_big.nlargest(10, "ef_total_gha")[["area", "ef_total_gha"]].reset_index(drop=True)
bot_abs = latest_big[latest_big["ef_total_gha"] > 0].nsmallest(10, "ef_total_gha")[["area", "ef_total_gha"]].reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.barh(top_abs["area"][::-1], top_abs["ef_total_gha"][::-1] / 1e9, color="#d62728", edgecolor="white")
ax.set_xlabel("Billion gha")
ax.set_title("Top 10 (largest total footprint)")
for i, v in enumerate(top_abs["ef_total_gha"][::-1] / 1e9):
    ax.text(v + 0.02, i, f"{v:.2f}", va="center", fontsize=9)

ax = axes[1]
ax.barh(bot_abs["area"][::-1], bot_abs["ef_total_gha"][::-1] / 1e6, color="#2ca02c", edgecolor="white")
ax.set_xlabel("Million gha")
ax.set_title("Bottom 10 (smallest total footprint)")
for i, v in enumerate(bot_abs["ef_total_gha"][::-1] / 1e6):
    ax.text(v + 0.02, i, f"{v:.1f}", va="center", fontsize=9)

fig.suptitle(f"Total Ecological Footprint ({YEAR})", fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

---
## 7. Component Breakdown: Top 10 Footprint Countries (absolute)

**What this shows:** A stacked bar chart decomposing each top-10 country's total EF into its 6 components, in absolute terms (billion gha). This reveals whether a country's footprint is driven primarily by carbon emissions, agricultural land use, or marine resource extraction.

**Components shown (stacked from bottom):**

| Component | Color | Formula | EQF |
|---|---|---|---|
| Cropland | Green | `Sum(CropProduction / WorldYield) x 2.1` | 2.1 |
| Grazing | Brown | `Sum(Heads x PastureDM) / 2.45 x 0.5` | 0.5 |
| Forest | Blue | `Roundwood_m3 / 1.81 x 1.3` | 1.3 |
| Fishing | Cyan | `PPR / MarineYield x 0.4` | 0.4 |
| Built-up | Grey | `BuiltUpArea_ha x 2.2` | 2.2 |
| Carbon | Red | `CO₂_t x 0.65 / 2.68 x 1.3` | 1.3 |

**Input data:** `data/overshoot_results.csv` — columns: `ef_cropland_gha`, `ef_grazing_gha`, `ef_forest_gha`, `ef_fishing_gha`, `ef_built_up_gha`, `ef_carbon_gha`.

In [ ]:
top10 = latest_big.nlargest(10, "ef_total_gha").copy()
comp_cols = [f"ef_{c}_gha" for c in components]

fig, ax = plt.subplots(figsize=(14, 6))

x = range(len(top10))
bottom = np.zeros(len(top10))
for col, label, color in zip(comp_cols, labels, colors_ef):
    vals = top10[col].values / 1e9
    ax.bar(x, vals, bottom=bottom, label=label, color=color, edgecolor="white", width=0.7)
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels(top10["area"].values, rotation=35, ha="right")
ax.set_ylabel("Billion gha")
ax.set_title(f"Footprint Component Breakdown — Top 10 Countries ({YEAR})", fontweight="bold")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

---
## 8. Component Breakdown: Top 10 per Capita

**What this shows:** Same stacked bar decomposition as Section 7, but normalised to per-capita values (gha/person). This removes the population effect, highlighting which EF components drive each high-consuming country's per-person footprint.

**Formula:** For each component: `EF_component_per_capita = ef_{component}_gha / (population x 1000)`.

The sum of all 6 per-capita components equals `ef_per_capita_gha`.

**Components (same color coding as Section 7):**
- Cropland (green), Grazing (brown), Forest (blue), Fishing (cyan), Built-up (grey), Carbon (red)

**Input data:** `data/overshoot_results.csv` — columns: `ef_cropland_gha` through `ef_carbon_gha`, `population` (thousands). Division by `population x 1000` converts to per-person values.

In [ ]:
top10_pc = latest_big.nlargest(10, "ef_per_capita_gha").copy()

fig, ax = plt.subplots(figsize=(14, 6))

x = range(len(top10_pc))
bottom = np.zeros(len(top10_pc))
pop = top10_pc["population"].values * 1000  # convert to actual people
for col, label, color in zip(comp_cols, labels, colors_ef):
    vals = top10_pc[col].values / pop
    ax.bar(x, vals, bottom=bottom, label=label, color=color, edgecolor="white", width=0.7)
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels(top10_pc["area"].values, rotation=35, ha="right")
ax.set_ylabel("gha per person")
ax.set_title(f"Per-Capita Footprint Breakdown — Top 10 Countries ({YEAR})", fontweight="bold")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

---
## 9. Top 10 & Bottom 10: Carbon Footprint per Capita

**What this shows:** Countries with the highest and lowest per-person carbon EF, measured in gha/person.

**Formula:** `Carbon EF per capita = ef_carbon_gha / (population x 1000)`

where `ef_carbon_gha = CO₂_emissions_t x (1 - ocean_fraction) / (AFCS x 44/12) x EQF_forest`:
- **CO₂_emissions_t** — territorial CO₂ from energy use (FAOSTAT Emissions domain), converted from kt to tonnes
- **ocean_fraction = 0.35** — 35% of CO₂ is absorbed by the ocean (Le Quere et al. 2018)
- **AFCS = 0.73 t C/ha/yr** — forest carbon uptake rate (IPCC 2006)
- **44/12 = 3.667** — molecular weight ratio to convert carbon to CO₂
- **EQF_forest = 1.3** — converts forest hectares to gha

**Input data:** `data/overshoot_results.csv` — column: `ef_carbon_gha`; source file: `data/06_carbon/co2_energy_faostat.csv`.

In [ ]:
latest_big["carbon_per_capita"] = latest_big["ef_carbon_gha"] / (latest_big["population"] * 1000)

top_carbon, bot_carbon = plot_top_bottom(latest_big, "carbon_per_capita",
                                          "Carbon Footprint per Capita",
                                          top_color="#d62728", bot_color="#2ca02c")

---
## 10. Top 10 & Bottom 10: Cropland Footprint per Capita

**What this shows:** Countries with the highest and lowest per-person cropland EF, measured in gha/person.

**Formula:** `Cropland EF per capita = ef_cropland_gha / (population x 1000)`

where `ef_cropland_gha = Sum_over_crops(Production_tonnes / WorldYield_t_per_ha) x EQF_cropland`:
- **Production_tonnes** — national crop production by commodity (FAOSTAT Production domain, Element: "Production")
- **WorldYield_t_per_ha** — world total production / world total harvested area, computed separately for each crop group
- **EQF_cropland = 2.1** — cropland is 2.1x as productive as the world-average hectare

**Input data:** `data/overshoot_results.csv` — column: `ef_cropland_gha`; source file: `data/01_cropland/crop_production.csv` (FAOSTAT bulk download: Production_Crops_Livestock_E_All_Data).

In [ ]:
latest_big["cropland_per_capita"] = latest_big["ef_cropland_gha"] / (latest_big["population"] * 1000)

top_crop, bot_crop = plot_top_bottom(latest_big, "cropland_per_capita",
                                      "Cropland Footprint per Capita",
                                      top_color="#8c564b", bot_color="#2ca02c")

---
## 11. Top 10 & Bottom 10: Fishing Grounds Footprint per Capita

**What this shows:** Countries with the highest and lowest per-person fishing EF, measured in gha/person. This quantifies the marine area needed to sustain each country's fish production.

**Formula:** `Fishing EF per capita = ef_fishing_gha / (population x 1000)`

where `ef_fishing_gha` is computed via the **PPR (Primary Production Required) method**:
1. For each FBS fish category, compute: `PPR = Catch_t x DR x (1/TE)^(TL-1) / WC`
   - **DR (Discard Rate)** = 1.27 — accounts for bycatch discards (Sea Around Us)
   - **TE (Transfer Efficiency)** = 0.1013 — energy transfer per trophic level (Pauly & Christensen 1995)
   - **TL (Trophic Level)** — average for each FBS category (e.g., Demersal Fish = 3.8, Pelagic Fish = 3.0, Crustaceans = 2.5)
   - **WC (Wet-to-Carbon ratio)** = 9.0 — converts wet-weight catch to carbon equivalent
2. Sum PPR across categories, divide by marine yield, multiply by `EQF_fishing (0.4)`

**Input data:** `data/overshoot_results.csv` — column: `ef_fishing_gha`; source file: `data/04_fishing/fish_supply_fbs.csv` (FAOSTAT FBS domain, Element: "Production", Unit: "1000 tonnes").

In [ ]:
latest_big["fishing_per_capita"] = latest_big["ef_fishing_gha"] / (latest_big["population"] * 1000)

top_fish, bot_fish = plot_top_bottom(latest_big, "fishing_per_capita",
                                      "Fishing Grounds Footprint per Capita",
                                      top_color="#17becf", bot_color="#2ca02c")

---
## 12. Top 10 & Bottom 10: Forest Product Footprint per Capita

**What this shows:** Countries with the highest and lowest per-person forest product EF, measured in gha/person. This quantifies the forest area needed to sustain each country's roundwood harvest.

**Formula:** `Forest EF per capita = ef_forest_gha / (population x 1000)`

where `ef_forest_gha = Roundwood_production_m3 / NAI x EQF_forest`:
- **Roundwood_production_m3** — national roundwood production (FAOSTAT Forestry Production domain, Item: "Roundwood", Element: "Production", Unit: m³)
- **NAI (Net Annual Increment)** = 1.81 m³/ha/yr — world-average forest growth rate (FAO Global Forest Resources Assessment 2020)
- **EQF_forest = 1.3** — forest land is 1.3x as productive as the world-average hectare

**Input data:** `data/overshoot_results.csv` — column: `ef_forest_gha`; source file: `data/03_forest/timber_production.csv`.

In [ ]:
latest_big["forest_per_capita"] = latest_big["ef_forest_gha"] / (latest_big["population"] * 1000)

top_forest, bot_forest = plot_top_bottom(latest_big, "forest_per_capita",
                                          "Forest Product Footprint per Capita",
                                          top_color="#1f77b4", bot_color="#2ca02c")

---
## 13. EF vs BC Scatter Plot

**What this shows:** Each country plotted by its per-capita biocapacity (x-axis) vs per-capita ecological footprint (y-axis). The dashed diagonal is the sustainability line where EF = BC.

**Interpretation:**
- **Above the diagonal** = ecological deficit (EF > BC): the country demands more than its territory regenerates
- **Below the diagonal** = ecological reserve (EF < BC): the country lives within its ecological means
- **Bubble size** = proportional to national population (larger bubbles = more people)
- **Bubble color** = ecological deficit per capita (red = deep deficit, green = large reserve), using a diverging colormap

**Formula (axes):**
- x-axis: `BC per capita = BC_total_gha / (population x 1000)`
- y-axis: `EF per capita = EF_total_gha / (population x 1000)`

**Input data:** `data/overshoot_results.csv` — columns: `ef_per_capita_gha`, `bc_per_capita_gha`, `population`, `deficit_per_capita` (derived). Major economies are labeled for reference.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

sc = ax.scatter(latest_big["bc_per_capita_gha"],
                latest_big["ef_per_capita_gha"],
                s=latest_big["population"] / 50,
                alpha=0.6, edgecolors="white", linewidth=0.5,
                c=latest_big["deficit_per_capita"],
                cmap="RdYlGn", vmin=-10, vmax=10)

# Diagonal line: EF = BC
lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
ax.plot([0, lim], [0, lim], "k--", alpha=0.4, label="EF = BC (sustainability line)")

# Label major countries
highlight = ["United States of America", "China", "India", "Brazil", "Russia",
             "Germany", "Japan", "Indonesia", "Australia", "Canada", "Nigeria"]
for _, row in latest_big[latest_big["area"].isin(highlight)].iterrows():
    name = row["area"].replace("United States of America", "USA")\
                       .replace("Russian Federation", "Russia")
    ax.annotate(name, (row["bc_per_capita_gha"], row["ef_per_capita_gha"]),
                fontsize=8, ha="left", va="bottom",
                xytext=(5, 3), textcoords="offset points")

ax.set_xlabel("Biocapacity per capita (gha/person)")
ax.set_ylabel("Ecological Footprint per capita (gha/person)")
ax.set_title(f"EF vs Biocapacity per Capita ({YEAR})\nBubble size = population, color = deficit",
             fontweight="bold")
ax.set_xlim(0, 20)
ax.set_ylim(0, 20)
ax.legend(loc="upper left")
plt.colorbar(sc, ax=ax, label="Deficit per capita (gha)", shrink=0.7)
fig.tight_layout()
plt.show()

---
## 14. Time Series: Major Economies

**What this shows:** Per-capita EF and BC trends (2014-2023) for 8 major economies: USA, China, India, Brazil, Russia, Germany, Japan, Indonesia. The shaded area between curves highlights the deficit (red) or reserve (green).

**Formula (per year per country):**
- `EF/cap = EF_total_gha / (population x 1000)` — red line ("EF/cap")
- `BC/cap = BC_total_gha / (population x 1000)` — green line ("BC/cap")

**What drives changes over time:**
- EF changes from shifts in energy consumption (carbon), agricultural production, forestry, and fisheries
- BC changes from land-use conversion (deforestation/reforestation), agricultural yield improvements (YF), and population growth (denominator)

**Input data:** `data/overshoot_results.csv` — full time series filtered by country name; columns: `ef_per_capita_gha`, `bc_per_capita_gha`, `year`.

In [ ]:
major = ["United States of America", "China", "India", "Brazil",
         "Russian Federation", "Germany", "Japan", "Indonesia"]
short_names = {"United States of America": "USA", "Russian Federation": "Russia"}

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for country, ax in zip(major[:4], axes.flat):
    cdf = df[df["area"] == country].sort_values("year")
    name = short_names.get(country, country)
    ax.plot(cdf["year"], cdf["ef_per_capita_gha"], "o-", color="#d62728", label="EF/cap")
    ax.plot(cdf["year"], cdf["bc_per_capita_gha"], "s-", color="#2ca02c", label="BC/cap")
    ax.fill_between(cdf["year"], cdf["bc_per_capita_gha"], cdf["ef_per_capita_gha"],
                    where=cdf["ef_per_capita_gha"] > cdf["bc_per_capita_gha"],
                    alpha=0.15, color="#d62728")
    ax.fill_between(cdf["year"], cdf["bc_per_capita_gha"], cdf["ef_per_capita_gha"],
                    where=cdf["ef_per_capita_gha"] <= cdf["bc_per_capita_gha"],
                    alpha=0.15, color="#2ca02c")
    ax.set_title(name, fontweight="bold")
    ax.set_ylabel("gha/person")
    ax.legend(fontsize=9)

fig.suptitle("EF vs BC per Capita Over Time — Major Economies", fontsize=14, fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for country, ax in zip(major[4:], axes.flat):
    cdf = df[df["area"] == country].sort_values("year")
    name = short_names.get(country, country)
    ax.plot(cdf["year"], cdf["ef_per_capita_gha"], "o-", color="#d62728", label="EF/cap")
    ax.plot(cdf["year"], cdf["bc_per_capita_gha"], "s-", color="#2ca02c", label="BC/cap")
    ax.fill_between(cdf["year"], cdf["bc_per_capita_gha"], cdf["ef_per_capita_gha"],
                    where=cdf["ef_per_capita_gha"] > cdf["bc_per_capita_gha"],
                    alpha=0.15, color="#d62728")
    ax.fill_between(cdf["year"], cdf["bc_per_capita_gha"], cdf["ef_per_capita_gha"],
                    where=cdf["ef_per_capita_gha"] <= cdf["bc_per_capita_gha"],
                    alpha=0.15, color="#2ca02c")
    ax.set_title(name, fontweight="bold")
    ax.set_ylabel("gha/person")
    ax.legend(fontsize=9)

fig.suptitle("EF vs BC per Capita Over Time — Major Economies (cont.)", fontsize=14, fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

---
## 15. Debtors vs Creditors: All Countries

**What this shows:** Two views of the global distribution of ecological deficit:
1. **Pie chart (left):** How many countries are in ecological deficit vs ecological reserve.
2. **Histogram (right):** Distribution of deficit per capita across all countries, clipped to [-15, +15] gha/person for readability.

**Formula:**
- `Ecological deficit = BC_total_gha - EF_total_gha` (country-level)
- `Deficit per capita = ecological_deficit_gha / (population x 1000)`
- Countries with deficit < 0 are "debtors"; countries with deficit >= 0 are "creditors"

**Interpretation:** In a typical year, roughly 80% of countries are ecological debtors. The histogram reveals that most debtors cluster around -1 to -5 gha/person, while a few creditors have very large per-capita reserves (e.g., >10 gha/person).

**Input data:** `data/overshoot_results.csv` — columns: `ecological_deficit_gha`, `population` (thousands); `deficit_per_capita` derived in this notebook.

In [ ]:
n_deficit = (latest_big["ecological_deficit_gha"] < 0).sum()
n_reserve = (latest_big["ecological_deficit_gha"] >= 0).sum()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie: debtor vs creditor count
ax = axes[0]
ax.pie([n_deficit, n_reserve], labels=[f"Deficit ({n_deficit})", f"Reserve ({n_reserve})"],
       colors=["#d62728", "#2ca02c"], autopct="%1.0f%%", startangle=90, textprops={"fontsize": 12})
ax.set_title(f"Countries in Deficit vs Reserve ({YEAR})", fontweight="bold")

# Histogram of deficit per capita
ax = axes[1]
vals = latest_big["deficit_per_capita"].clip(-15, 15)
ax.hist(vals, bins=40, color="#1f77b4", edgecolor="white", alpha=0.8)
ax.axvline(0, color="black", linewidth=1.2, linestyle="--")
ax.set_xlabel("Ecological deficit per capita (gha/person)")
ax.set_ylabel("Number of countries")
ax.set_title("Distribution of Ecological Deficit per Capita", fontweight="bold")
ax.annotate("DEFICIT", xy=(-10, ax.get_ylim()[1] * 0.8), fontsize=11, color="#d62728", fontweight="bold")
ax.annotate("RESERVE", xy=(3, ax.get_ylim()[1] * 0.8), fontsize=11, color="#2ca02c", fontweight="bold")

fig.tight_layout()
plt.show()

print(f"{n_deficit} countries in ecological deficit, {n_reserve} with ecological reserve")

---
## 16. Summary Tables

**What this shows:** Global-level summary statistics for the latest year and for all years in the dataset.

**Key metrics displayed:**
- **World EF** — sum of all countries' `ef_total_gha`, in billion gha
- **World BC** — sum of all countries' `bc_total_gha`, in billion gha
- **Overshoot** — `World EF - World BC`, in billion gha (positive = global deficit)
- **Number of Earths** — `World EF / World BC` (how many planet Earths humanity requires)
- **Overshoot Day** — `floor(365 x World BC / World EF)`, the calendar day by which humanity exhausts nature's annual budget

**Input data:** `data/overshoot_global_summary.csv` — all columns; computed by `scripts/calculate_overshoot.py` by aggregating the country-level results.

In [ ]:
print(f"\n{'='*70}")
print(f"GLOBAL SUMMARY ({YEAR})")
print(f"{'='*70}")
row = gs[gs["year"] == YEAR].iloc[0]
print(f"  World EF:          {row['world_ef_gha']/1e9:>8.2f} billion gha")
print(f"  World BC:          {row['world_bc_gha']/1e9:>8.2f} billion gha")
print(f"  Overshoot:         {row['overshoot_gha']/1e9:>8.2f} billion gha")
print(f"  Number of Earths:  {row['number_of_earths']:>8.2f}")
from datetime import datetime, timedelta
overshoot_date = datetime(YEAR, 1, 1) + timedelta(days=int(row['overshoot_day']) - 1)
print(f"  Overshoot Day:     {overshoot_date.strftime('%B %d')} (day {int(row['overshoot_day'])})")
print()

In [ ]:
print("Global Summary — All Years:")
display(gs.style.format({
    "world_ef_gha": "{:.2e}",
    "world_bc_gha": "{:.2e}",
    "overshoot_gha": "{:.2e}",
    "number_of_earths": "{:.3f}",
    "overshoot_day": "{:.0f}"
}))